<a href="https://colab.research.google.com/github/Arx15E/IntegracionDatos/blob/main/Proyecto%20final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# RETO FINAL — Integración de Datos y Prospectiva
# Gases de Efecto Invernadero y Eventos de Riesgo Climático Global
# ================================================================

# ================================================================
# CELDA 1 — Instalar librerías
# ================================================================
!pip install pandas numpy plotly scikit-learn -q

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas correctamente')


# ================================================================
# CELDA 2 — Cargar datos
# Sube los 4 archivos CSV al panel de archivos de Colab
# ================================================================

df_temp = pd.read_csv('/content/1- temperature-anomaly.csv')
df_co2  = pd.read_csv('/content/2- annual-co-emissions-by-region.csv')
df_cap  = pd.read_csv('/content/3- co-emissions-per-capita.csv')
df_dis  = pd.read_csv(
    '/content/_EmergencyEventsDatabase-CountryProfiles_emdat-country-profiles_2023_04_06.csv',
    sep=';', decimal=','
)

print(f'🌡️  Temperature anomaly:    {df_temp.shape}')
print(f'🏭  CO₂ emissions:          {df_co2.shape}')
print(f'👤  CO₂ per cápita:         {df_cap.shape}')
print(f'🌪️  Desastres (EM-DAT):     {df_dis.shape}')


# ================================================================
# CELDA 3 — Descripción Dataset 1: Temperatura Global
# ================================================================

print('=' * 60)
print('📘 DATASET 1: Anomalía de Temperatura Global (1850-2023)')
print('=' * 60)
print(f'\n📅 Rango temporal: {df_temp["Year"].min()} — {df_temp["Year"].max()}')
print(f'🌐 Entidades:      {df_temp["Entity"].unique().tolist()}')
print()
print('📐 Variables numéricas:')
print('   • Global average temperature anomaly relative to 1961-1990 (°C)')
print('   • Upper bound (IC 95%)')
print('   • Lower bound (IC 95%)')
print()
print('📊 Estadísticas descriptivas:')
display(df_temp.describe())
print()
print('❓ Valores nulos:')
print(df_temp.isnull().sum())


# ================================================================
# CELDA 4 — Descripción Dataset 2: Emisiones de CO₂
# ================================================================

print('=' * 60)
print('📘 DATASET 2: Emisiones Anuales de CO₂ por País (1750-2022)')
print('=' * 60)

# Filtrar solo países (tienen código ISO)
df_co2_paises = df_co2[df_co2['Code'].notna() & (df_co2['Code'] != '')].copy()

print(f'\n📅 Rango temporal: {df_co2_paises["Year"].min()} — {df_co2_paises["Year"].max()}')
print(f'🌐 Países únicos:  {df_co2_paises["Entity"].nunique()}')
print()
print('📐 Variables numéricas:')
print('   • Annual CO₂ emissions (toneladas)')
print()
print('📊 Estadísticas descriptivas:')
display(df_co2_paises[['Year','Annual CO₂ emissions']].describe())
print()
print('❓ Valores nulos:')
print(df_co2_paises.isnull().sum())


# ================================================================
# CELDA 5 — Descripción Dataset 3: Desastres Naturales (EM-DAT)
# ================================================================

print('=' * 60)
print('📘 DATASET 3: Desastres Naturales EM-DAT (1900-2023)')
print('=' * 60)
print(f'\n📅 Rango temporal: {df_dis["Year"].min()} — {df_dis["Year"].max()}')
print(f'🌐 Países únicos:  {df_dis["Country"].nunique()}')
print(f'🌀 Tipos de desastre disponibles:')
print(df_dis['Disaster Type'].value_counts().to_string())
print()
print('📐 Variables numéricas clave:')
print('   • Total Events   — número de eventos por país/año/tipo')
print('   • Total Deaths   — muertes totales')
print('   • Total Affected — personas afectadas')
print('   • Total Damage (USD, adjusted) — daños económicos ajustados')
print()
print('📊 Estadísticas descriptivas:')
display(df_dis[['Total Events','Total Deaths','Total Affected']].describe())
print()
print('❓ Valores nulos:')
print(df_dis.isnull().sum())


# ================================================================
# CELDA 6 — Limpieza y preparación
# ================================================================

# --- Dataset temperatura: solo anomalía Global ---
df_temp_clean = df_temp[df_temp['Entity'] == 'Global'][['Year','Global average temperature anomaly relative to 1961-1990']].copy()
df_temp_clean.columns = ['Year', 'Temp_Anomaly']
df_temp_clean = df_temp_clean.sort_values('Year').reset_index(drop=True)

# --- Dataset CO₂: solo países, renombrar columna ---
df_co2_clean = df_co2_paises[['Entity','Code','Year','Annual CO₂ emissions']].copy()
df_co2_clean.columns = ['Country','Code','Year','CO2_tons']
# Convertir a millones de toneladas
df_co2_clean['CO2_Mt'] = df_co2_clean['CO2_tons'] / 1e6

# --- Dataset desastres: solo eventos climáticos ---
tipos_climaticos = ['Flood', 'Storm', 'Drought', 'Extreme temperature ', 'Wildfire', 'Landslide']
df_dis_clean = df_dis[df_dis['Disaster Type'].isin(tipos_climaticos)].copy()
df_dis_clean = df_dis_clean[['Year','Country','ISO','Disaster Type',
                              'Total Events','Total Deaths','Total Affected',
                              'Total Damage (USD, adjusted)']].copy()
df_dis_clean.columns = ['Year','Country','ISO','Tipo_Desastre',
                        'Eventos','Muertos','Afectados','Daños_USD']
df_dis_clean[['Eventos','Muertos','Afectados','Daños_USD']] = \
    df_dis_clean[['Eventos','Muertos','Afectados','Daños_USD']].fillna(0)

print('✅ Temperature anomaly limpiado  —', df_temp_clean.shape)
print('✅ CO₂ por país limpiado         —', df_co2_clean.shape)
print('✅ Desastres climáticos limpiado —', df_dis_clean.shape)
print()
print('📅 Rango temporal en común: 1900 — 2022')


# ================================================================
# CELDA 7 — MODELO DE INTEGRACIÓN 1
#            Merge global por Año: temperatura + desastres globales
# ================================================================

# Agregar desastres a nivel global por año
df_dis_global = df_dis_clean.groupby('Year').agg(
    Eventos_Total  = ('Eventos',    'sum'),
    Muertos_Total  = ('Muertos',    'sum'),
    Afectados_Total= ('Afectados',  'sum'),
    Daños_Total    = ('Daños_USD',  'sum')
).reset_index()

# CO₂ global por año (suma de todos los países)
df_co2_global = df_co2_clean.groupby('Year')['CO2_Mt'].sum().reset_index()
df_co2_global.columns = ['Year','CO2_Global_Mt']

# Merge triple: CO₂ + Temperatura + Desastres por Año
df_modelo1 = df_co2_global \
    .merge(df_temp_clean, on='Year', how='inner') \
    .merge(df_dis_global,  on='Year', how='inner')

print('✅ MODELO 1 — Merge global por Año (CO₂ + Temperatura + Desastres)')
print(f'   Registros: {df_modelo1.shape[0]}')
print(f'   Rango:     {df_modelo1["Year"].min()} — {df_modelo1["Year"].max()}')
display(df_modelo1.head(5))


# ================================================================
# CELDA 8 — MODELO DE INTEGRACIÓN 2
#            Agregación por Década + Join: país × período
# ================================================================

# Década en desastres
df_dis_clean['Decada'] = (df_dis_clean['Year'] // 10) * 10

# Década en CO₂ por país
df_co2_clean['Decada'] = (df_co2_clean['Year'] // 10) * 10

# Agregar CO₂ promedio por país y década
df_co2_dec = df_co2_clean.groupby(['Country','Code','Decada']).agg(
    CO2_prom_Mt = ('CO2_Mt', 'mean')
).reset_index()

# Agregar desastres por país y década
df_dis_dec = df_dis_clean.groupby(['Country','ISO','Decada']).agg(
    Eventos_dec  = ('Eventos',  'sum'),
    Muertos_dec  = ('Muertos',  'sum'),
    Daños_dec    = ('Daños_USD','sum')
).reset_index()

# Join por País + Década
df_modelo2 = pd.merge(
    df_co2_dec,
    df_dis_dec,
    left_on =['Country','Code','Decada'],
    right_on=['Country','ISO', 'Decada'],
    how='inner'
).drop(columns='ISO')

print('✅ MODELO 2 — Agregación por Década + Join (País × Período)')
print(f'   Registros: {df_modelo2.shape[0]}')
print(f'   Países:    {df_modelo2["Country"].nunique()}')
print(f'   Décadas:   {sorted(df_modelo2["Decada"].unique())}')
display(df_modelo2.head(5))


# ================================================================
# CELDA 9 — Visualización: Tendencia global CO₂ + Temperatura + Desastres
# ================================================================

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        'Emisiones globales de CO₂ (Mt)',
        'Anomalía de temperatura global (°C vs 1961-1990)',
        'Eventos climáticos extremos por año'
    ),
    shared_xaxes=True,
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(
    x=df_modelo1['Year'], y=df_modelo1['CO2_Global_Mt'],
    fill='tozeroy', name='CO₂ (Mt)',
    line=dict(color='#D85A30', width=1.5)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_modelo1['Year'], y=df_modelo1['Temp_Anomaly'],
    name='Anomalía Temp (°C)',
    line=dict(color='#E24B4A', width=2),
    fill='tozeroy'
), row=2, col=1)

fig.add_trace(go.Bar(
    x=df_modelo1['Year'], y=df_modelo1['Eventos_Total'],
    name='Eventos climáticos',
    marker_color='#378ADD'
), row=3, col=1)

fig.update_layout(
    title_text='📈 Tendencia Global: CO₂ · Temperatura · Desastres Climáticos (1900-2022)',
    height=700, showlegend=True, hovermode='x unified'
)
fig.show()


# ================================================================
# CELDA 10 — Matriz de correlación
# ================================================================

cols_corr = ['CO2_Global_Mt','Temp_Anomaly','Eventos_Total','Muertos_Total','Afectados_Total']
corr = df_modelo1[cols_corr].corr()

etiquetas = {
    'CO2_Global_Mt'  : 'CO₂ (Mt)',
    'Temp_Anomaly'   : 'Anomalía Temp',
    'Eventos_Total'  : 'Nº Eventos',
    'Muertos_Total'  : 'Muertos',
    'Afectados_Total': 'Afectados'
}
corr_renamed = corr.rename(index=etiquetas, columns=etiquetas)

fig_corr = px.imshow(
    corr_renamed,
    title='🔗 Matriz de Correlación — GEI vs Impacto Climático',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f'
)
fig_corr.update_layout(width=600, height=500)
fig_corr.show()

print('\n📌 Interpretación:')
print('   Valores > 0.6  → correlación positiva fuerte')
print('   Valores < -0.6 → correlación negativa fuerte')
print('   Valores ~  0   → baja asociación lineal')


# ================================================================
# CELDA 11 — Tendencia por tipo de desastre y década
# ================================================================

eventos_tipo_dec = df_dis_clean.groupby(['Decada','Tipo_Desastre'])['Eventos'].sum().reset_index()

fig_tipo = px.bar(
    eventos_tipo_dec,
    x='Decada', y='Eventos', color='Tipo_Desastre',
    title='🌪️ Eventos Climáticos por Tipo y Década',
    labels={'Decada':'Década','Eventos':'Nº de Eventos','Tipo_Desastre':'Tipo'},
    barmode='stack'
)
fig_tipo.update_layout(height=480)
fig_tipo.show()


# ================================================================
# CELDA 12 — Top 15 países más afectados
# ================================================================

top15 = df_dis_clean.groupby('Country').agg(
    Eventos_tot  = ('Eventos',  'sum'),
    Muertos_tot  = ('Muertos',  'sum'),
    Afectados_tot= ('Afectados','sum')
).reset_index().sort_values('Eventos_tot', ascending=False).head(15)

fig_top = px.bar(
    top15.sort_values('Eventos_tot'),
    x='Eventos_tot', y='Country',
    orientation='h',
    title='🌐 Top 15 Países con Más Eventos Climáticos (1900-2023)',
    color='Muertos_tot',
    color_continuous_scale='Reds',
    labels={'Eventos_tot':'Nº Eventos','Country':'País','Muertos_tot':'Muertos'}
)
fig_top.update_layout(height=520)
fig_top.show()


# ================================================================
# CELDA 13 — CO₂ por década: top emisores
# ================================================================

top_emisores = df_co2_dec.groupby('Country')['CO2_prom_Mt'].mean() \
    .sort_values(ascending=False).head(10).index.tolist()

df_top_em = df_co2_dec[df_co2_dec['Country'].isin(top_emisores)]

fig_em = px.line(
    df_top_em, x='Decada', y='CO2_prom_Mt', color='Country',
    title='🏭 Emisiones Promedio de CO₂ por Década — Top 10 Países',
    labels={'Decada':'Década','CO2_prom_Mt':'CO₂ promedio (Mt)','Country':'País'}
)
fig_em.update_layout(height=480)
fig_em.show()


# ================================================================
# CELDA 14 — Regresión lineal: CO₂ global → Nº de eventos climáticos
# ================================================================

df_reg = df_modelo1[['CO2_Global_Mt','Eventos_Total']].dropna()
X = df_reg[['CO2_Global_Mt']].values
y = df_reg['Eventos_Total'].values

modelo_reg = LinearRegression()
modelo_reg.fit(X, y)
y_pred = modelo_reg.predict(X)

r2  = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)

print('📐 REGRESIÓN LINEAL — CO₂ Global → Eventos Climáticos')
print(f'   Ecuación:    Eventos = {modelo_reg.coef_[0]:.4f} × CO₂ + {modelo_reg.intercept_:.2f}')
print(f'   R²:          {r2:.4f}')
print(f'   MAE:         {mae:.2f} eventos')
print()
if r2 > 0.7:
    print('   ✅ El modelo explica bien la variación en eventos climáticos')
elif r2 > 0.4:
    print('   ⚠️  El modelo explica parcialmente la variación')
else:
    print('   ℹ️  La relación lineal es débil — hay otros factores en juego')

fig_reg = px.scatter(
    df_reg, x='CO2_Global_Mt', y='Eventos_Total',
    opacity=0.6, title='📉 Regresión: CO₂ Global vs Eventos Climáticos',
    labels={'CO2_Global_Mt':'CO₂ Global (Mt)','Eventos_Total':'Nº Eventos Climáticos'}
)
fig_reg.add_trace(go.Scatter(
    x=sorted(df_reg['CO2_Global_Mt']),
    y=modelo_reg.predict(np.array(sorted(df_reg['CO2_Global_Mt'])).reshape(-1,1)),
    mode='lines', name=f'Regresión lineal (R²={r2:.3f})',
    line=dict(color='red', width=2.5)
))
fig_reg.show()


# ================================================================
# CELDA 15 — PROSPECTIVA: Simulación Monte Carlo 2025-2045
# ================================================================

np.random.seed(42)
n_sim       = 10_000
años_futuro = np.arange(2025, 2046)
co2_base    = df_modelo1['CO2_Global_Mt'].iloc[-1]  # último valor registrado

escenarios = {
    '🟢 Acuerdo de París (-2%/año)' : -0.02,
    '🟡 Tendencia actual (+1.5%/año)':  0.015,
    '🔴 Sin acción (+3%/año)'        :  0.03,
}
colores = ['#3B6D11', '#BA7517', '#A32D2D']

fig_mc = go.Figure()

resultados_2045 = {}

for (nombre, tasa), color in zip(escenarios.items(), colores):
    proyecciones = []
    for año_i, año in enumerate(años_futuro):
        co2_proy = co2_base * ((1 + tasa) ** (año_i + 1))
        co2_sim  = np.random.normal(co2_proy, co2_proy * 0.12, n_sim)
        ev_sim   = np.clip(modelo_reg.predict(co2_sim.reshape(-1, 1)), 0, None)
        proyecciones.append({
            'año': año,
            'p50': np.percentile(ev_sim, 50),
            'p05': np.percentile(ev_sim,  5),
            'p95': np.percentile(ev_sim, 95)
        })

    df_proy = pd.DataFrame(proyecciones)
    resultados_2045[nombre] = df_proy[df_proy['año'] == 2045]['p50'].values[0]

    fig_mc.add_trace(go.Scatter(
        x=df_proy['año'], y=df_proy['p50'],
        name=nombre, mode='lines',
        line=dict(color=color, width=2.5)
    ))
    fig_mc.add_trace(go.Scatter(
        x=list(df_proy['año']) + list(df_proy['año'][::-1]),
        y=list(df_proy['p95']) + list(df_proy['p05'][::-1]),
        fill='toself', fillcolor=color, opacity=0.12,
        line=dict(color='rgba(0,0,0,0)'), showlegend=False
    ))

fig_mc.update_layout(
    title='🔮 Proyección Monte Carlo — Eventos Climáticos 2025-2045<br>'
          '<sup>Banda = IC 90% | n = 10.000 simulaciones por año</sup>',
    xaxis_title='Año',
    yaxis_title='Eventos Climáticos Proyectados (por año)',
    height=520, hovermode='x unified'
)
fig_mc.show()

print('\n📊 Eventos climáticos proyectados para 2045:')
for nombre, val in resultados_2045.items():
    print(f'   {nombre}: {val:.0f} eventos/año')


# ================================================================
# CELDA 16 — Herramienta interactiva: exploración por tipo de desastre
# ================================================================

from ipywidgets import interact, Dropdown, Output
import ipywidgets as widgets

out = Output()
tipos = sorted(df_dis_clean['Tipo_Desastre'].unique().tolist())

def explorar(tipo):
    with out:
        out.clear_output()
        df_f = df_dis_clean[df_dis_clean['Tipo_Desastre'] == tipo]
        dec_f = df_f.groupby('Decada').agg(
            Eventos = ('Eventos','sum'),
            Muertos = ('Muertos','sum')
        ).reset_index()

        print(f'\n🔍 Tipo de desastre: {tipo}')
        print(f'   Eventos totales registrados: {int(df_f["Eventos"].sum()):,}')
        print(f'   Muertos totales:             {int(df_f["Muertos"].sum()):,}')
        print(f'   Afectados totales:           {int(df_f["Afectados"].sum()):,}')
        print(f'   Daños totales (USD adj.):    ${df_f["Daños_USD"].sum():,.0f}')

        top5 = df_f.groupby('Country')['Eventos'].sum().nlargest(5).reset_index()
        print(f'\n🏴 Top 5 países con más eventos de tipo "{tipo}":')
        display(top5)

        fig1 = px.bar(
            dec_f, x='Decada', y='Eventos',
            title=f'Eventos de {tipo} por Década',
            color='Eventos', color_continuous_scale='Blues'
        )
        fig1.show()

        fig2 = px.bar(
            dec_f, x='Decada', y='Muertos',
            title=f'Muertos por {tipo} por Década',
            color='Muertos', color_continuous_scale='Reds'
        )
        fig2.show()

tipo_selector = Dropdown(options=tipos, value=tipos[0], description='Tipo:')

print('🎛️ Herramienta Interactiva — Exploración por Tipo de Desastre Climático')
interact(explorar, tipo=tipo_selector)
display(out)

✅ Librerías cargadas correctamente
🌡️  Temperature anomaly:    (522, 6)
🏭  CO₂ emissions:          (30308, 4)
👤  CO₂ per cápita:         (26600, 4)
🌪️  Desastres (EM-DAT):     (10431, 13)
📘 DATASET 1: Anomalía de Temperatura Global (1850-2023)

📅 Rango temporal: 1850 — 2023
🌐 Entidades:      ['Global', 'Northern hemisphere', 'Southern hemisphere']

📐 Variables numéricas:
   • Global average temperature anomaly relative to 1961-1990 (°C)
   • Upper bound (IC 95%)
   • Lower bound (IC 95%)

📊 Estadísticas descriptivas:


,Code,Year,Global average temperature anomaly relative to 1961-1990,Upper bound of the annual temperature anomaly (95% confidence interval),Lower bound of the annual temperature anomaly (95% confidence interval)
count,0.0,522.000000,522.000000,522.000000,522.000000
mean,NaN,1936.500000,-0.072792,0.038905,-0.184489
std,NaN,50.276825,0.387320,0.347909,0.432559
min,NaN,1850.000000,-0.701569,-0.486698,-0.947684
25%,NaN,1893.000000,-0.354566,-0.199500,-0.521331
50%,NaN,1936.500000,-0.193912,-0.056851,-0.294819
75%,NaN,1980.000000,0.110119,0.208315,0.052181
max,NaN,2023.000000,1.275727,1.333305,1.236863



❓ Valores nulos:
Entity                                                                       0
Code                                                                       522
Year                                                                         0
Global average temperature anomaly relative to 1961-1990                     0
Upper bound of the annual temperature anomaly (95% confidence interval)      0
Lower bound of the annual temperature anomaly (95% confidence interval)      0
dtype: int64
📘 DATASET 2: Emisiones Anuales de CO₂ por País (1750-2022)

📅 Rango temporal: 1750 — 2022
🌐 Países únicos:  217

📐 Variables numéricas:
   • Annual CO₂ emissions (toneladas)

📊 Estadísticas descriptivas:


,Year,Annual CO₂ emissions
count,24157.000000,2.415700e+04
mean,1948.949124,1.449600e+08
std,59.996632,1.348178e+09
min,1750.000000,0.000000e+00
25%,1918.000000,1.612160e+05
50%,1965.000000,2.407248e+06
75%,1994.000000,2.175466e+07
max,2022.000000,3.714979e+10



❓ Valores nulos:
Entity                  0
Code                    0
Year                    0
Annual CO₂ emissions    0
dtype: int64
📘 DATASET 3: Desastres Naturales EM-DAT (1900-2023)

📅 Rango temporal: 1900 — 2023
🌐 Países únicos:  225
🌀 Tipos de desastre disponibles:
Disaster Type
Flood                    3837
Storm                    2761
Earthquake               1087
Drought                   784
Landslide                 652
Extreme temperature       565
Wildfire                  374
Volcanic activity         231
Insect infestation         92
Mass movement (dry)        44
Glacial lake outburst       2
Fog                         1
Animal accident             1

📐 Variables numéricas clave:
   • Total Events   — número de eventos por país/año/tipo
   • Total Deaths   — muertes totales
   • Total Affected — personas afectadas
   • Total Damage (USD, adjusted) — daños económicos ajustados

📊 Estadísticas descriptivas:


...


📌 Interpretación:
   Valores > 0.6  → correlación positiva fuerte
   Valores < -0.6 → correlación negativa fuerte
   Valores ~  0   → baja asociación lineal


📐 REGRESIÓN LINEAL — CO₂ Global → Eventos Climáticos
   Ecuación:    Eventos = 0.0055 × CO₂ + -48.75
   R²:          0.8967
   MAE:         30.29 eventos

   ✅ El modelo explica bien la variación en eventos climáticos



📊 Eventos climáticos proyectados para 2045:
   🟢 Acuerdo de París (-2%/año): 217 eventos/año
   🟡 Tendencia actual (+1.5%/año): 507 eventos/año
   🔴 Sin acción (+3%/año): 707 eventos/año
🎛️ Herramienta Interactiva — Exploración por Tipo de Desastre Climático


interactive(children=(Dropdown(description='Tipo:', options=('Drought', 'Extreme temperature ', 'Flood', 'Land…

Output()